# 2차 복원: 영상 3편 64장 + 점별 신뢰도 저장 (Colab MCP로 실행한 셀)

먼저 **런타임 → 런타임 유형 변경 → T4 GPU**. 4번 셀에서 "파일 선택" 버튼으로 `frames_multi.zip`(main 24 · low 20 · top 20)을 올립니다.
결과 `points_multi.npz`(xyz, rgb, conf, cams, view_id)는 로컬 `tools/build.py --npz points_multi.npz --conf-min 2 ...`로 정리합니다.

In [ ]:
# 1. GPU 확인
import torch, time, sys, os
T0 = time.time()
assert torch.cuda.is_available(), "GPU 가 없습니다. 런타임 유형을 T4 GPU 로 바꾸세요."
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability()
print(f"{p.name} / {p.total_memory/1e9:.1f}GB / 계산 능력 {cap[0]}.{cap[1]} / bf16 {cap[0] >= 8}")

In [ ]:
# 2. 설치 (torch 는 건드리지 않음) + 경로 끼우기
import subprocess
t = time.time()
subprocess.run("git clone -q --depth 1 https://github.com/facebookresearch/map-anything.git /content/map-anything", shell=True)
r = subprocess.run("cd /content/map-anything && pip install -q -e .", shell=True, capture_output=True, text=True)
print(r.stderr[-1500:])
sys.path.insert(0, '/content/map-anything')
import mapanything
print("설치", round(time.time()-t), "초")

In [ ]:
# 3. 모델 준비 (4.91GB)
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
t = time.time()
model = MapAnything.from_pretrained("facebook/map-anything-apache").to("cuda").eval()
T_MODEL = time.time() - t
print(f"모델 준비 {T_MODEL:.0f}초")

In [ ]:
# 4. 프레임 올리기 → 셀 아래 "파일 선택" 버튼으로 frames_multi.zip 을 고르세요
import zipfile, glob
from google.colab import files
up = files.upload()
ZIP = os.path.join('/content', list(up.keys())[0])
os.makedirs('/content/frames', exist_ok=True)
for f in glob.glob('/content/frames/*'): os.remove(f)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if n.lower().endswith('.jpg'):
            open(os.path.join('/content/frames', os.path.basename(n)), 'wb').write(z.read(n))
paths = sorted(glob.glob('/content/frames/*.jpg'))
print(len(paths), "장", round(os.path.getsize(ZIP)/1e6, 1), "MB", {t: sum(os.path.basename(p).startswith(t) for p in paths) for t in ('main', 'low', 'top')})

In [ ]:
# 5. 복원 (T4 는 반드시 fp16)
views = load_images(paths)
IMG_SHAPE = tuple(views[0]['img'].shape[-2:])
print("모델 입력 크기", IMG_SHAPE)
torch.cuda.reset_peak_memory_stats()
t = time.time()
pred = model.infer(views, memory_efficient_inference=True, minibatch_size=1,
                   use_amp=True, amp_dtype="fp16",
                   apply_mask=True, mask_edges=True, apply_confidence_mask=False)
torch.cuda.synchronize()
T_INFER = time.time() - t
PEAK = torch.cuda.max_memory_allocated() / 1e9
print(f"복원 {T_INFER:.1f}초, 최대 GPU 메모리 {PEAK:.2f}GB")

In [ ]:
# 6. 점·색·신뢰도·카메라 저장 (신뢰도는 로컬에서 기준을 바꿔 가며 거른다)
import numpy as np, json
print("출력 키:", sorted(pred[0].keys()))
pts, cols, confs, cams, per_view = [], [], [], [], []
for v in pred:
    m = v['mask'][0, ..., 0].cpu().numpy()
    pts.append(v['pts3d'][0].cpu().numpy()[m])
    cols.append((v['img_no_norm'][0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)[m])
    c = v['conf'][0].float().cpu().numpy()
    confs.append((c[..., 0] if c.ndim == 3 else c)[m].astype(np.float16))
    cams.append(v['camera_poses'][0].cpu().numpy())
    per_view.append(int(m.sum()))
xyz = np.concatenate(pts).astype(np.float32); rgb = np.concatenate(cols); conf = np.concatenate(confs)
cams = np.stack(cams).astype(np.float32)
view_id = np.concatenate([np.full(n, i, np.uint8) for i, n in enumerate(per_view)])
N_RAW = len(xyz)
lo = np.percentile(xyz, 0.5, axis=0); hi = np.percentile(xyz, 99.5, axis=0)
keep = np.all((xyz > lo) & (xyz < hi), axis=1)
xyz, rgb, conf, view_id = xyz[keep], rgb[keep], conf[keep], view_id[keep]
cpos = cams[:, :3, 3]
spread = float(np.linalg.norm(cpos.max(0) - cpos.min(0)))
diag = float(np.linalg.norm(xyz.max(0) - xyz.min(0)))
print(f"점 {N_RAW:,} -> 튄 점 제거 후 {len(xyz):,}")
print(f"카메라 퍼짐 {spread:.3f} / 장면 대각선 {diag:.3f} = {spread/diag:.4f}")
print("신뢰도 백분위 5/25/50/75/95:", np.percentile(conf.astype(np.float32), [5, 25, 50, 75, 95]).round(3))
metrics = dict(gpu=p.name, frames=len(paths), frame_names=[os.path.basename(x) for x in paths],
               model_input_hw=list(IMG_SHAPE), t_model_s=round(T_MODEL, 1), t_infer_s=round(T_INFER, 2),
               peak_gpu_gb=round(PEAK, 2), points_raw=N_RAW, points_kept=int(len(xyz)), points_per_view=per_view,
               cam_spread=spread, scene_diag=diag, spread_ratio=spread/diag)
np.savez_compressed('/content/points_multi.npz', xyz=xyz, rgb=rgb, conf=conf, cams=cams, view_id=view_id)
json.dump(metrics, open('/content/metrics_multi.json', 'w'), indent=1, ensure_ascii=False)
print(f"points_multi.npz {os.path.getsize('/content/points_multi.npz')/1e6:.1f}MB")

In [ ]:
# 7. 내려받기 (브라우저가 여러 파일 다운로드 허용을 물으면 허용)
files.download('/content/metrics_multi.json')
files.download('/content/points_multi.npz')
print("전 과정", round(time.time()-T0), "초")